<a href="https://colab.research.google.com/github/raaga102005/python-automation-scripts/blob/main/smart_data_cleaner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import re
import os

# ── STEP 1: Create messy sample data ─────────────────────
data = {
    'name':       ['  alice sharma ', 'BOB KUMAR', 'charlie d',
                       'Diana Nair', '  eve  ', 'frank', None],
    'email':      ['alice@email.com', 'bob@@broken', 'charlie@test.com',
                                              'Diana@Email.COM', '', 'frank@work.in', 'ghost@co.in'],
    'phone':      ['9876543210', '+91-8765432109', '91 7654 321 098',
                                                                     'invalid', '6543210987', '+91-5432109876', '9012345678'],
    'salary':     ['₹45,000', 'Rs. 62000', '38000', '₹71,500',
                                                                                            'N/A', '₹55,000', '48000'],
    'department': ['Tech', 'tech', 'TECH', 'Marketing',
                                                                                                                   'marketing', 'HR', 'hr'],
    'score':      [85, 92, np.nan, 78, 95, np.nan, 60]
}

df = pd.DataFrame(data)
print("ORIGINAL DATA:")
print(df)
print(f"\nShape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")

# ── STEP 2: Clean names ───────────────────────────────────
df['name'] = df['name'].fillna('Unknown')
df['name'] = df['name'].str.strip().str.title()

# ── STEP 3: Clean and validate emails ────────────────────
email_pat = r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$'
df['email'] = df['email'].str.lower().str.strip()
df['email_valid'] = df['email'].apply(
    lambda x: bool(re.match(email_pat, str(x)))
    )

# ── STEP 4: Clean phone numbers ──────────────────────────
def clean_phone(p):
    digits = re.sub(r'\D', '', str(p))
    return digits[-10:] if len(digits) >= 10 else None

df['phone_clean'] = df['phone'].apply(clean_phone)

# ── STEP 5: Clean salary ─────────────────────────────────
def clean_salary(s):
    cleaned = re.sub(r'[^\d.]', '', str(s))
    return float(cleaned) if cleaned else None

df['salary_clean'] = df['salary'].apply(clean_salary)

# ── STEP 6: Standardise department ───────────────────────
df['department'] = df['department'].str.strip().str.title()

# ── STEP 7: Fill missing scores with department average ──
df['score'] = df.groupby('department')['score'].transform(
    lambda x: x.fillna(x.mean())
    )

# ── STEP 8: NumPy analysis on scores ─────────────────────
scores = df['score'].dropna().values
print(f"\nSCORE ANALYSIS:")
print(f"Mean:   {np.mean(scores):.1f}")
print(f"Median: {np.median(scores):.1f}")
print(f"Std:    {np.std(scores):.1f}")
print(f"Min:    {np.min(scores):.1f}")
print(f"Max:    {np.max(scores):.1f}")

# Performance categories using np.where
df['performance'] = np.where(df['score'] >= 90, 'Excellent',
                           np.where(df['score'] >= 75, 'Good',
                                                    np.where(df['score'] >= 60, 'Average', 'Below Average')))

# ── STEP 9: Department summary ────────────────────────────
print("\nDEPARTMENT SUMMARY:")
summary = df.groupby('department').agg(
    headcount=('name', 'count'),
    avg_salary=('salary_clean', 'mean'),
    avg_score=('score', 'mean')
    ).round(1)
print(summary)

# ── STEP 10: Save cleaned data ────────────────────────────
clean_cols = ['name', 'email', 'email_valid', 'phone_clean',
              'salary_clean', 'department', 'score', 'performance']
df_clean = df[clean_cols]
df_clean.to_csv('cleaned_data.csv', index=False)
print("\nCleaned data saved to cleaned_data.csv")
print("\nFINAL CLEAN DATA:")
print(df_clean)

ORIGINAL DATA:
              name             email            phone     salary department  \
0    alice sharma    alice@email.com       9876543210    ₹45,000       Tech   
1        BOB KUMAR       bob@@broken   +91-8765432109  Rs. 62000       tech   
2        charlie d  charlie@test.com  91 7654 321 098      38000       TECH   
3       Diana Nair   Diana@Email.COM          invalid    ₹71,500  Marketing   
4            eve                           6543210987        N/A  marketing   
5            frank     frank@work.in   +91-5432109876    ₹55,000         HR   
6             None       ghost@co.in       9012345678      48000         hr   

   score  
0   85.0  
1   92.0  
2    NaN  
3   78.0  
4   95.0  
5    NaN  
6   60.0  

Shape: (7, 6)
Missing values:
name          1
email         0
phone         0
salary        0
department    0
score         2
dtype: int64

SCORE ANALYSIS:
Mean:   79.8
Median: 85.0
Std:    13.5
Min:    60.0
Max:    95.0

DEPARTMENT SUMMARY:
            headcount